In [1]:
!pip install remfile dandi pynwb h5py scikit-learn matplotlib numpy

In [ ]:
import remfile
import h5py
from pynwb import NWBHDF5IO
from dandi.dandiapi import DandiAPIClient
import numpy as np

# Get all NWB files from DANDI
client = DandiAPIClient()
dandiset = client.get_dandiset("000402", "draft")
all_assets = list(dandiset.get_assets())
nwb_assets = [a for a in all_assets if a.path.endswith('.nwb')]

print()

# Process each file
for i, asset in enumerate(nwb_assets, 1):
    print(f"File {i}/{len(nwb_assets)}: {asset.path}")
    
    # Load NWB file
    s3_url = asset.get_content_url(follow_redirects=1, strip_query=True)
    rf = remfile.File(s3_url)
    h5 = h5py.File(rf, "r")
    io = NWBHDF5IO(file=h5, load_namespaces=True)
    nwb = io.read()
    
    # For first file only, print detailed metadata
    if i == 1:
        print("\n" + "="*70)
        print("DETAILED METADATA FOR FIRST FILE")
        print("="*70)
        
        if hasattr(nwb, 'intervals'):
            for interval_name in nwb.intervals.keys():
                interval = nwb.intervals[interval_name]
                print(f"\nInterval: {interval_name}")
                print(f"Total entries: {len(interval)}")
                print(f"Columns: {interval.colnames}")
                
                # Print all metadata for each entry
                print(f"\nAll data in {interval_name}:")
                for col in interval.colnames:
                    data = interval[col][:]
                    print(f"\n  Column: {col}")
                    print(f"  Type: {type(data)}")
                    if hasattr(data, 'shape'):
                        print(f"  Shape: {data.shape}")
                    if hasattr(data, 'dtype'):
                        print(f"  Dtype: {data.dtype}")
                    
                    # Show first 5 values
                    if len(data) > 0:
                        print(f"  First 5 values:")
                        for idx in range(min(5, len(data))):
                            print(f"    [{idx}]: {data[idx]}")
                    
                    # If short_movie_name, show unique values
                    if col == 'short_movie_name':
                        unique = np.unique(data)
                        print(f"  Unique values: {unique}")
        
        print("\n" + "="*70)
        print("SUMMARY FOR REMAINING FILES")
        print("="*70 + "\n")
    
    # Print intervals summary for all files
    if hasattr(nwb, 'intervals'):
        for interval_name in nwb.intervals.keys():
            interval = nwb.intervals[interval_name]
            n_entries = len(interval)
            
            print(f"  {interval_name}: {n_entries} entries")
            
            # Print stimulus types if available
            if 'short_movie_name' in interval.colnames:
                stim_types = np.array(interval['short_movie_name'][:])
                unique_types = np.unique(stim_types)
                for stim_type in unique_types:
                    count = np.sum(stim_types == stim_type)
                    print(f"    - {stim_type}: {count}")
    
    # Print ROI series info
    if hasattr(nwb, 'processing') and 'ophys' in nwb.processing:
        ophys = nwb.processing['ophys']
        if 'Fluorescence' in ophys.data_interfaces:
            fluorescence = ophys.data_interfaces['Fluorescence']
            roi_series = fluorescence.roi_response_series
            print(f"  ROI series: {len(roi_series)} planes")
            for rs_name in roi_series.keys():
                rs = roi_series[rs_name]
                n_neurons = rs.data.shape[1]
                n_timepoints = rs.data.shape[0]
                print(f"    {rs_name}: {n_neurons} neurons, {n_timepoints} timepoints")
    
    print()


File 1/19: sub-17797/sub-17797_ses-4-scan-7_behavior+image+ophys.nwb

DETAILED METADATA FOR FIRST FILE

Interval: Clip
Total entries: 384
Columns: ('start_time', 'stop_time', 'stimulus_type', 'condition_hash', 'movie_name', 'short_movie_name', 'duration')

All data in Clip:

  Column: start_time
  Type: <class 'numpy.ndarray'>
  Shape: (384,)
  Dtype: float64
  First 5 values:
    [0]: 18.85249095557191
    [1]: 28.91899018881776
    [2]: 38.98549681303956
    [3]: 49.051991754750986
    [4]: 59.11848669646241

  Column: stop_time
  Type: <class 'numpy.ndarray'>
  Shape: (384,)
  Dtype: float64
  First 5 values:
    [0]: 28.81900411246278
    [1]: 38.88550191519715
    [2]: 48.95199709532716
    [3]: 59.01849632857301
    [4]: 69.08498888609864

  Column: stimulus_type
  Type: <class 'numpy.ndarray'>
  Shape: (384,)
  Dtype: object
  First 5 values:
    [0]: stimulus.Clip
    [1]: stimulus.Clip
    [2]: stimulus.Clip
    [3]: stimulus.Clip
    [4]: stimulus.Clip

  Column: condition_h